<a href="https://colab.research.google.com/github/bijelic02/myRAGProject/blob/main/MyRAGProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Connect to Google Drive

In [17]:
#helper for multiple runnings
import subprocess
subprocess.run(['fusermount', '-u', '/content/drive'], capture_output=True)
print("Unmounted")

Unmounted


In [18]:
from google.colab import drive
drive.mount('/content/drive')
print("Mounted successfully")

ValueError: Mountpoint must not already contain files

This mounts your Google Drive into Colab. When you run it, it asks for permission once and then your Drive is accessible at `/content/drive/MyDrive/`. Your notebook file itself (`.ipynb`) gets saved to Drive automatically if you set it up right.

**Save the notebook itself to Drive**

In Colab go to:
```
File → Save a copy in Drive
```

This moves your notebook from Colab's temporary storage into your actual Google Drive. From that point on, `Ctrl+S` saves it there. Never work from the default "Colab Notebooks" auto-created file without doing this first.
---

**Also save to GitHub (highly recommended)**

The spec specifically asks for a GitHub repo as a deliverable anyway, so set it up early:
```
File → Save a copy in GitHub


Install dependencies

In [ ]:
!pip install transformers torch accelerate bitsandbytes \
             sentence-transformers \
             faiss-cpu chromadb \
             rank-bm25 \
             langchain langchain-community nltk \
             gradio \
             pypdf beautifulsoup4 requests \
             wikipedia-api \
             pandas numpy tqdm scikit-learn -q

Check if everything is actually installed correctly

In [ ]:
import transformers
import torch
import sentence_transformers
import faiss
import chromadb
import gradio
import langchain
from rank_bm25 import BM25Okapi
import pandas as pd
import sklearn
import wikipediaapi
import json
import os
from datetime import datetime

print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
print("sentence_transformers:", sentence_transformers.__version__)
print("faiss version: OK")
print("chromadb:", chromadb.__version__)
print("gradio:", gradio.__version__)
print("langchain:", langchain.__version__)
print("pandas:", pandas.__version__)
print("GPU available:", torch.cuda.is_available())

Define documents list with metadata

In [ ]:
DOCUMENTS = [
    {
        "title": "Rodent",
        "url": "https://en.wikipedia.org/wiki/Rodent",
        "category": "overview",
        "topic": "rodents"
    },
    {
        "title": "List of rodents",
        "url": "https://en.wikipedia.org/wiki/List_of_rodents",
        "category": "overview",
        "topic": "rodents"
    },
    {
        "title": "Beaver",
        "url": "https://en.wikipedia.org/wiki/Beaver",
        "category": "species",
        "topic": "rodents"
    },
    {
        "title": "Nutria",
        "url": "https://en.wikipedia.org/wiki/Nutria",
        "category": "species",
        "topic": "rodents"
    },
    {
        "title": "Gopher",
        "url": "https://en.wikipedia.org/wiki/Gopher",
        "category": "species",
        "topic": "rodents"
    },
    {
        "title": "Chinchilla",
        "url": "https://en.wikipedia.org/wiki/Chinchilla",
        "category": "species",
        "topic": "rodents"
    },
    {
        "title": "Squirrel",
        "url": "https://en.wikipedia.org/wiki/Squirrel",
        "category": "species",
        "topic": "rodents"
    },
    {
        "title": "Capybara",
        "url": "https://en.wikipedia.org/wiki/Capybara",
        "category": "species",
        "topic": "rodents"
    },
    {
        "title": "Guinea pig",
        "url": "https://en.wikipedia.org/wiki/Guinea_pig",
        "category": "species",
        "topic": "rodents"
    },
    {
        "title": "Marmot",
        "url": "https://en.wikipedia.org/wiki/Marmot",
        "category": "species",
        "topic": "rodents"
    },
    {
        "title": "Pika",
        "url": "https://en.wikipedia.org/wiki/Pika",
        "category": "species",
        "topic": "rodents"
    },
    {
        "title": "Rabbit",
        "url": "https://en.wikipedia.org/wiki/Rabbit",
        "category": "species",
        "topic": "rodents"
    },
    {
        "title": "Rat",
        "url": "https://en.wikipedia.org/wiki/Rat",
        "category": "species",
        "topic": "rodents"
    },
]

print(f"Defined {len(DOCUMENTS)} documents")

Fetch and normalize documents

In [ ]:
def fetch_wikipedia_page(wiki, title):
    """Fetch a single Wikipedia page and return its text."""
    page = wiki.page(title)
    if not page.exists():
        print(f"  WARNING: Page '{title}' not found")
        return None
    return page.text

In [ ]:
def normalize_text(text):
    """Basic text normalization."""
    # Remove excessive newlines
    lines = text.split('\n')
    # Filter out very short lines (usually section headers or empty)
    lines = [line.strip() for line in lines if len(line.strip()) > 30]
    return ' '.join(lines)

In [ ]:
def ingest_documents(document_list, save_path):
    """Fetch all documents from Wikipedia and save with metadata."""

    # Initialize Wikipedia API
    wiki = wikipediaapi.Wikipedia(
        language='en',
        user_agent='RodentRAGProject/1.0'
    )

    ingested = []
    failed = []

    for doc in document_list:
        print(f"Fetching: {doc['title']}...")

        try:
            raw_text = fetch_wikipedia_page(wiki, doc['title'])

            if raw_text is None:
                failed.append(doc['title'])
                continue

            clean_text = normalize_text(raw_text)

            # Build the full document object with metadata
            document = {
                "id": doc['title'].lower().replace(' ', '_'),
                "title": doc['title'],
                "text": clean_text,
                "metadata": {
                    "source": doc['url'],
                    "category": doc['category'],
                    "topic": doc['topic'],
                    "date_ingested": datetime.now().strftime("%Y-%m-%d"),
                    "char_count": len(clean_text),
                    "word_count": len(clean_text.split())
                }
            }

            ingested.append(document)
            print(f"  OK — {document['metadata']['word_count']} words")

        except Exception as e:
            print(f"  FAILED: {e}")
            failed.append(doc['title'])

    # Save to JSON file on Drive
    os.makedirs(save_path, exist_ok=True)
    output_file = os.path.join(save_path, 'documents.json')

    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(ingested, f, indent=2, ensure_ascii=False)

    print(f"\n{'='*40}")
    print(f"Successfully ingested: {len(ingested)} documents")
    print(f"Failed: {len(failed)} documents {failed if failed else ''}")
    print(f"Saved to: {output_file}")

    return ingested



In [ ]:
# Run ingestion - saves to Google Drive
DOCS_SAVE_PATH = "/content/drive/MyDrive/rag_project/documents"


In [ ]:
documents = ingest_documents(DOCUMENTS, DOCS_SAVE_PATH)

Preview what you fetched

In [ ]:
def preview_documents(docs):
    """Print a summary table of all ingested documents."""

    rows = []
    for doc in docs:
        rows.append({
            "ID": doc['id'],
            "Title": doc['title'],
            "Category": doc['metadata']['category'],
            "Words": doc['metadata']['word_count'],
            "Chars": doc['metadata']['char_count'],
            "Date Ingested": doc['metadata']['date_ingested']
        })

    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    print(f"\nTotal words across all documents: {df['Words'].sum():,}")
    print(f"Average words per document: {df['Words'].mean():.0f}")

In [ ]:


preview_documents(documents)

Load documents back from Drive (for future sessions)

In [ ]:
def load_documents(save_path):
    """Load previously ingested documents from Drive."""
    output_file = os.path.join(save_path, 'documents.json')

    if not os.path.exists(output_file):
        print("No saved documents found. Run ingestion first.")
        return []

    with open(output_file, 'r', encoding='utf-8') as f:
        docs = json.load(f)

    print(f"Loaded {len(docs)} documents from {output_file}")
    return docs

# In future sessions, instead of re-fetching from Wikipedia,
# just call this:
# documents = load_documents(DOCS_SAVE_PATH)

In [ ]:
import os

path = "/content/drive/MyDrive/rag_project/documents/documents.json"

if os.path.exists(path):
    size = os.path.getsize(path)
    print(f"File exists!")
    print(f"File size: {size:,} bytes ({size/1024:.1f} KB)")
else:
    print("File not found - something went wrong")